In [1]:
import collections
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# =====================================================================
# 1. Mock FASTQ Data Generator (For Demonstration)
# =====================================================================
# We will write some dummy FASTQ files to simulate the input data
def generate_mock_fastq(filename, sequences):
    with open(filename, 'w') as f:
        for i, seq in enumerate(sequences):
            f.write(f"@READ_{i}\n")
            f.write(f"{seq}\n")
            f.write("+\n")
            f.write("I" * len(seq) + "\n")

# Mock samples with sequences (Karyotypes: XX = 0, XY = 1)
# We'll use a short sequence length and a k-mer size of 3 for demonstration
mock_samples_data = [
    ("sample_XX_1.fastq", ["ACACAC", "ACACAC", "TGTGTG"], 0),
    ("sample_XX_2.fastq", ["ACACAC", "TGTGTG", "TGTGTG"], 0),
    ("sample_XY_1.fastq", ["GGGGGG", "GGGGGG", "CCCCCC"], 1),
    ("sample_XY_2.fastq", ["GGGGGG", "CCCCCC", "CCCCCC"], 1)
]

for filename, seqs, _ in mock_samples_data:
    generate_mock_fastq(filename, seqs)

# =====================================================================
# 2. FASTQ Parser & K-mer Counter
# =====================================================================
def read_fastq_sequences(filepath):
    """Reads a FASTQ file and extracts the sequences."""
    sequences = []
    with open(filepath, 'r') as f:
        lines = f.readlines()
        # FASTQ files have 4 lines per record; sequence is the 2nd line
        for i in range(1, len(lines), 4):
            sequences.append(lines[i].strip())
    return sequences

def get_kmers_from_sequence(sequence, k):
    """Splits a sequence into kmers of length k."""
    return [sequence[i:i+k] for i in range(len(sequence) - k + 1)]

def build_kmer_vocabulary(filepaths, k):
    """Extracts all unique kmers from all files to build a global vocabulary."""
    vocab = set()
    for filepath in filepaths:
        for seq in read_fastq_sequences(filepath):
            vocab.update(get_kmers_from_sequence(seq, k))
    return sorted(list(vocab))

# Define parameters
K_SIZE = 3
filepaths = [item[0] for item in mock_samples_data]
labels = [item[2] for item in mock_samples_data]

# Build a mapping from k-mer to feature index
kmer_vocab = build_kmer_vocabulary(filepaths, K_SIZE)
kmer_to_idx = {kmer: idx for idx, kmer in enumerate(kmer_vocab)}
print(f"K-mer Vocabulary ({len(kmer_vocab)} unique kmers): {kmer_vocab}\n")

# =====================================================================
# 3. PyTorch Dataset and DataLoader
# =====================================================================
class KmerKaryotypeDataset(Dataset):
    def __init__(self, filepaths, labels, kmer_to_idx, k):
        self.filepaths = filepaths
        self.labels = labels
        self.kmer_to_idx = kmer_to_idx
        self.k = k
        self.vocab_size = len(kmer_to_idx)

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        filepath = self.filepaths[idx]
        label = self.labels[idx]

        # Count kmers in this sample
        counts = collections.Counter()
        sequences = read_fastq_sequences(filepath)
        for seq in sequences:
            kmers = get_kmers_from_sequence(seq, self.k)
            counts.update(kmers)

        # Convert counts to a dense feature vector
        feature_vector = torch.zeros(self.vocab_size, dtype=torch.float32)
        for kmer, count in counts.items():
            if kmer in self.kmer_to_idx:
                feature_vector[self.kmer_to_idx[kmer]] = count

        return feature_vector, torch.tensor(label, dtype=torch.long)

dataset = KmerKaryotypeDataset(filepaths, labels, kmer_to_idx, K_SIZE)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

# =====================================================================
# 4. PyTorch Classifier Model
# =====================================================================
class KmerClassifier(nn.Module):
    def __init__(self, vocab_size, num_classes):
        super(KmerClassifier, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(vocab_size, 8),
            nn.ReLU(),
            nn.Linear(8, num_classes)
        )

    def forward(self, x):
        return self.network(x)

# Instantiate the model, loss, and optimizer
model = KmerClassifier(vocab_size=len(kmer_vocab), num_classes=2)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# =====================================================================
# 5. Model Training Loop
# =====================================================================
print("Starting Model Training...")
model.train()
for epoch in range(20):
    total_loss = 0
    for features, targets in dataloader:
        optimizer.zero_grad()
        outputs = model(features)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}/20], Loss: {total_loss:.4f}")

# Simple evaluation on our mock data
model.eval()
with torch.no_grad():
    print("\nPredictions:")
    for features, targets in dataloader:
        outputs = model(features)
        _, predicted = torch.max(outputs.data, 1)
        for f, t, p in zip(features, targets, predicted):
            print(f"True Label: {t.item()}, Predicted Label: {p.item()}")


K-mer Vocabulary (6 unique kmers): ['ACA', 'CAC', 'CCC', 'GGG', 'GTG', 'TGT']

Starting Model Training...
Epoch [5/20], Loss: 0.6485
Epoch [10/20], Loss: 0.2542
Epoch [15/20], Loss: 0.0580
Epoch [20/20], Loss: 0.0145

Predictions:
True Label: 1, Predicted Label: 1
True Label: 0, Predicted Label: 0
True Label: 0, Predicted Label: 0
True Label: 1, Predicted Label: 1


### Download Real 1000 Genomes FASTQ Files
We will download a small subset of low-coverage FASTQ reads from the 1000 Genomes Project for an XX individual (e.g., HG00096) and an XY individual (e.g., HG00097) to train and evaluate our model.

In [5]:
import urllib.request
import gzip
import os

# We use verified ENA mirror URLs for stable access to these fastq files
urls = {
    "HG00096_XX.fastq.gz": "https://ftp.sra.ebi.ac.uk/vol1/fastq/SRR062/SRR062634/SRR062634_1.fastq.gz",
    "HG00097_XY.fastq.gz": "https://ftp.sra.ebi.ac.uk/vol1/fastq/SRR062/SRR062635/SRR062635_1.fastq.gz"
}

for name, url in urls.items():
    local_gz = f"/content/{name}"
    local_fastq = local_gz.replace(".gz", "")

    if not os.path.exists(local_fastq):
        print(f"Downloading {name}...")
        # Download first 8MB to get sufficient reads for feature training
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as response, open(local_gz, 'wb') as out_file:
            out_file.write(response.read(8 * 1024 * 1024))

        # Unzip
        print(f"Unzipping {name}...")
        with gzip.open(local_gz, 'rb') as f_in, open(local_fastq, 'wb') as f_out:
            # Limit to 40,000 lines (10,000 reads) for speed in Colab
            for _ in range(40000):
                line = f_in.readline()
                if not line: break
                f_out.write(line)
        os.remove(local_gz)

print("FASTQ download and preparation complete!")

Unzipping HG00097_XY.fastq.gz...
FASTQ download and preparation complete!


### Build Large K-mer Vocabulary & Model for Real Data
Now we will parse the sequences using $k = 31$, identify the top distinct k-mers, and build our PyTorch dataset classifier.

In [6]:
import collections
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

K_SIZE_REAL = 31
real_samples = [
    ("/content/HG00096_XX.fastq", 0),  # XX
    ("/content/HG00097_XY.fastq", 1)   # XY
]

# Extract k-mers from the real data to form our feature space
print("Analyzing k-mers from real samples...")
all_kmer_counts = collections.Counter()

for filepath, _ in real_samples:
    seqs = read_fastq_sequences(filepath)
    for seq in seqs:
        if 'N' in seq: continue  # Skip ambiguous reads
        kmers = get_kmers_from_sequence(seq, K_SIZE_REAL)
        all_kmer_counts.update(kmers)

# Select the top 1000 most common k-mers to keep the feature tensor size practical
most_common_kmers = [kmer for kmer, count in all_kmer_counts.most_common(1000)]
real_kmer_to_idx = {kmer: idx for idx, kmer in enumerate(most_common_kmers)}

print(f"Created feature vocabulary with {len(real_kmer_to_idx)} unique {K_SIZE_REAL}-mers.")

Analyzing k-mers from real samples...
Created feature vocabulary with 1000 unique 31-mers.


In [7]:
# Prepare PyTorch datasets
real_filepaths = [item[0] for item in real_samples]
real_labels = [item[1] for item in real_samples]

real_dataset = KmerKaryotypeDataset(real_filepaths, real_labels, real_kmer_to_idx, K_SIZE_REAL)
real_dataloader = DataLoader(real_dataset, batch_size=2, shuffle=True)

# Define the neural network for our real features
class RealKmerClassifier(nn.Module):
    def __init__(self, vocab_size, num_classes):
        super(RealKmerClassifier, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(vocab_size, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        # Normalize feature vector to prevent scaling issues with high counts
        x_norm = nn.functional.normalize(x, p=2, dim=1)
        return self.network(x_norm)

real_model = RealKmerClassifier(vocab_size=len(real_kmer_to_idx), num_classes=2)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(real_model.parameters(), lr=0.005)

# Train model
real_model.train()
print("Training classifier on real data features...")
for epoch in range(50):
    for features, targets in real_dataloader:
        optimizer.zero_grad()
        outputs = real_model(features)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/50], Loss: {loss.item():.4f}")

# Check the classification of our real genomes
real_model.eval()
with torch.no_grad():
    print("\nFinal Evaluation on Real Samples:")
    for features, targets in real_dataloader:
        outputs = real_model(features)
        _, predicted = torch.max(outputs.data, 1)
        for f, t, p in zip(features, targets, predicted):
            label_map = {0: "XX (Female)", 1: "XY (Male)"}
            print(f"Sample: True Karyotype: {label_map[t.item()]} | Predicted: {label_map[p.item()]}")

Training classifier on real data features...
Epoch [10/50], Loss: 0.5079
Epoch [20/50], Loss: 0.3380
Epoch [30/50], Loss: 0.1167
Epoch [40/50], Loss: 0.0492
Epoch [50/50], Loss: 0.0317

Final Evaluation on Real Samples:
Sample: True Karyotype: XX (Female) | Predicted: XX (Female)
Sample: True Karyotype: XY (Male) | Predicted: XY (Male)


### Download External Test Samples (HG00099 & HG00100)
We will download sequencing reads for two new individuals to serve as our independent validation test set.

In [8]:
import urllib.request
import gzip
import os

test_urls = {
    "HG00099_XX_test.fastq.gz": "https://ftp.sra.ebi.ac.uk/vol1/fastq/SRR062/SRR062637/SRR062637_1.fastq.gz",
    "HG00100_XY_test.fastq.gz": "https://ftp.sra.ebi.ac.uk/vol1/fastq/SRR062/SRR062641/SRR062641_1.fastq.gz"
}

for name, url in test_urls.items():
    local_gz = f"/content/{name}"
    local_fastq = local_gz.replace(".gz", "")

    if not os.path.exists(local_fastq):
        print(f"Downloading {name}...")
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as response, open(local_gz, 'wb') as out_file:
            out_file.write(response.read(8 * 1024 * 1024))

        print(f"Unzipping {name}...")
        with gzip.open(local_gz, 'rb') as f_in, open(local_fastq, 'wb') as f_out:
            for _ in range(40000):
                line = f_in.readline()
                if not line: break
                f_out.write(line)
        os.remove(local_gz)

print("Test FASTQ files downloaded and prepared!")

Unzipping HG00099_XX_test.fastq.gz...
Unzipping HG00100_XY_test.fastq.gz...
Test FASTQ files downloaded and prepared!


### Evaluate Trained Model on External Test Samples

In [9]:
test_samples = [
    ("/content/HG00099_XX_test.fastq", 0),  # XX Female
    ("/content/HG00100_XY_test.fastq", 1)   # XY Male
]

test_filepaths = [item[0] for item in test_samples]
test_labels = [item[1] for item in test_samples]

# Instantiate the dataset using the pre-existing training k-mer dictionary mapping
test_dataset = KmerKaryotypeDataset(test_filepaths, test_labels, real_kmer_to_idx, K_SIZE_REAL)
test_dataloader = DataLoader(test_dataset, batch_size=2, shuffle=False)

real_model.eval()
with torch.no_grad():
    print("\n--- Generalization Performance on Unseen Test Samples ---")
    for features, targets in test_dataloader:
        outputs = real_model(features)
        _, predicted = torch.max(outputs.data, 1)
        for f, t, p in zip(features, targets, predicted):
            label_map = {0: "XX (Female)", 1: "XY (Male)"}
            print(f"Unseen Sample: True Karyotype: {label_map[t.item()]} | Predicted: {label_map[p.item()]}")


--- Generalization Performance on Unseen Test Samples ---
Unseen Sample: True Karyotype: XX (Female) | Predicted: XX (Female)
Unseen Sample: True Karyotype: XY (Male) | Predicted: XX (Female)
